# Offline Futures Research Pipeline Demo

This notebook explains the full repository pipeline end to end using a compact notebook subset.

Research question: can futures term-structure and lagged volatility features help forecast forward realized variance across ES, CL, and GC?

All code paths shown here are offline-first and read local Parquet files from `data/raw`.


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

from futures_roll_overlay.data_access.raw_cache import load_config
from futures_roll_overlay.data_access.raw_cache import list_raw_inventory
from futures_roll_overlay.pipeline.run_research_pipeline import run_research_pipeline

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
config = load_config(PROJECT_ROOT / "config.toml")
print("project_root", PROJECT_ROOT)
print("assets", config["research"]["assets"])

## 1) Offline Contract And Raw Inventory

Inputs for default runs are local Parquet files under `data/raw`.

The manifest stores table-level metadata: asset symbol, date range, row count, schema columns, and file size.

This is the explicit offline portability contract for the project.


In [ ]:
inventory = list_raw_inventory(config=config)
display(
    inventory[
        [
            "asset",
            "table",
            "file",
            "start_date",
            "end_date",
            "row_count",
            "file_size_bytes",
        ]
    ].sort_values(["table", "asset"])
)
print("total payload bytes", int(inventory["file_size_bytes"].sum()))

## 2) Run The Production Pipeline Once

We run the same production entrypoint used in CLI mode.

The run writes full artifacts and notebook-focused subset artifacts:
- `dataset.parquet`, `predictions.parquet`, `metrics.csv`, `feature_importance.csv`
- `notebook_dataset.parquet`, `notebook_predictions.parquet`

The subset files keep this notebook readable while still covering the full workflow.


In [ ]:
run_result = run_research_pipeline(config_path=PROJECT_ROOT / "config.toml")
run_dir = Path(run_result["run_dir"])
print("run_dir", run_dir)
print("files")
for path in sorted(run_dir.iterdir()):
    print("-", path.name)

## 3) Continuous Futures, Returns, And Realized Variance Target

The dataset columns encode the core target construction.

Definitions:
- $r_t$: daily log return on day $t$.
- $rv_t = r_t^2$: daily realized variance.
- $RV_{t,t+H} = \sum_{i=1}^{H} r_{t+i}^2$: forward $H$-day realized variance.
- $RV^{ann}_{t,t+H} = (A/H)RV_{t,t+H}$ with annualization factor $A=252$.

In this project, the model target is `target_forward_rv_annualized`.


In [ ]:
notebook_dataset = pd.read_parquet(run_dir / "notebook_dataset.parquet")
display(notebook_dataset.head(10))
print("rows", len(notebook_dataset))
print("assets", notebook_dataset["asset"].value_counts().to_dict())
print("date range", notebook_dataset["date"].min(), notebook_dataset["date"].max())

## 4) Feature Engineering Layer

Features are intentionally explainable:
- term-structure shape (`spread`, `roll_yield`, `slope`)
- regime indicators (`regime_*`)
- lagged variance and lagged returns (`lag_rv_1`, `lag_log_return_1`)

This keeps the story interview-defensible and avoids opaque model inputs.


In [ ]:
feature_columns = [
    column
    for column in notebook_dataset.columns
    if column
    not in {
        "date",
        "asset",
        "target_forward_rv_annualized",
        "forward_realized_variance",
    }
]
print("feature columns")
for name in feature_columns:
    print("-", name)
display(notebook_dataset.groupby("asset")[feature_columns].mean().round(4))

## 5) Walk-Forward Evaluation Protocol

Evaluation is run per asset with ordered train/test windows, then aggregated.

That design keeps index semantics simple and avoids cross-asset leakage.

Models compared:
- persistence baseline,
- ridge regression.


In [ ]:
metrics = pd.read_csv(run_dir / "metrics.csv")
display(metrics)
pooled = metrics[metrics["scope"] == "pooled"].copy()
display(pooled[["model", "mse", "rmse", "mae", "r2"]].sort_values("rmse"))

## 6) Prediction Diagnostics

We inspect predictions against realized forward variance and check residual behavior.

For the subset, the visual and residual summary are enough to explain whether errors are centered or biased.


In [ ]:
notebook_predictions = pd.read_parquet(run_dir / "notebook_predictions.parquet")
display(notebook_predictions.head())
figure, axis = plt.subplots(figsize=(9, 5), dpi=160, constrained_layout=True)
for model_name, model_slice in notebook_predictions.groupby("model"):
    axis.scatter(
        model_slice["actual"],
        model_slice["prediction"],
        s=20,
        alpha=0.65,
        label=model_name,
    )
diagonal_min = min(
    notebook_predictions["actual"].min(), notebook_predictions["prediction"].min()
)
diagonal_max = max(
    notebook_predictions["actual"].max(), notebook_predictions["prediction"].max()
)
axis.plot(
    [diagonal_min, diagonal_max],
    [diagonal_min, diagonal_max],
    color="black",
    linewidth=1.0,
)
axis.set_title("Notebook Subset: Predicted vs Realized Forward Variance")
axis.set_xlabel("Actual")
axis.set_ylabel("Prediction")
axis.legend()
plt.show()

residual_summary = (
    notebook_predictions.assign(
        residual=lambda frame: frame["actual"] - frame["prediction"]
    )
    .groupby("model")["residual"]
    .describe()
)
display(residual_summary)

## 7) Interpretation Layer

The ridge coefficient table provides a direct interpretation of model direction and magnitude.

Large absolute coefficients indicate stronger contribution to the variance forecast in standardized feature space.

This is the explainability output expected for the project.


In [ ]:
feature_importance = pd.read_csv(run_dir / "feature_importance.csv")
feature_importance = feature_importance.sort_values(
    "coefficient", key=lambda s: s.abs(), ascending=False
)
display(feature_importance)

figure, axis = plt.subplots(figsize=(9, 4), dpi=160, constrained_layout=True)
axis.bar(feature_importance["feature"], feature_importance["coefficient"])
axis.axhline(0.0, color="black", linewidth=1.0)
axis.set_title("Ridge Coefficients (Standardized Features)")
axis.set_ylabel("Coefficient")
axis.tick_params(axis="x", rotation=45)
plt.show()

## 8) Limitations And Refresh Path

Current notebook and demo settings use a compact subset for readability.

Limits:
- short horizon and simple models,
- no nonlinear model family,
- no transaction-cost trading backtest (out of scope by design).

Optional one-time raw refresh from ClickHouse is available via:
`uv run python -m futures_roll_overlay.data_access.refresh_clickhouse_raw`

That refresh updates `data/raw` only; the default research and notebook flows remain offline-first.


In [ ]:
summary = {
    "run_dir": str(run_dir),
    "n_dataset_rows": int(len(notebook_dataset)),
    "n_prediction_rows": int(len(notebook_predictions)),
    "models": sorted(notebook_predictions["model"].unique().tolist()),
    "assets": sorted(notebook_dataset["asset"].unique().tolist()),
}
print(json.dumps(summary, indent=2))